# Tomahawk P2: plasma stability with BOUT++ (elm-pb), pressure x2 and x3: n-scan, Spitzer resistivity, nx 260

**Settings:** Internet **ON**, accelerator **None (CPU)**. **No inputs needed.** Use **Save Version -> Save & Run All (Commit)**.
Run P1-P4 at the same time (each one is independent).

**What it is:** AuroraNova's Tomahawk N9A2 equilibrium (enlarged TF48, axisymmetric, SHA-256 `8e6b295e4f8dbfe7...`, embedded) is turned
into a BOUT++ grid (closed flux surfaces psi_n 0.40-1.00, orthogonal mesh), and BOUT++ 5.1.1's elm-pb model (3-field reduced MHD,
peeling-ballooning) computes the linear growth rate of each toroidal mode n. Pressure is scaled (x1 = design) to find the
ballooning margin. Density 3.46e20 m^-3 (N9A2 axis pressure at Ti = Te = 12 keV), D-T ion mass 2.5.

**Time:** BOUT++ build about 15-25 min, then about 4.8 h of runs (estimate; capped at 11.3 h, cases that don't fit are reported).

**Output:** `Tomahawk_Plasma_P2_Results.zip`: per case growth_rate.json, mode.png, ParaView .vts files, BOUT.inp/settings/log; summary.json; growth_vs_n.png.
Open the .vts files in ParaView (File -> Open, or run OPEN_IN_PARAVIEW.py from ParaView's Python Shell).

**If any case fails, the notebook raises an error so Kaggle shows the run as failed.**

## 1. Unpack code and equilibrium

In [ ]:
import base64, io, json, os, shutil, subprocess, sys, tarfile, time, zipfile, hashlib
from pathlib import Path
print("NOTEBOOK VERSION: Tomahawk plasma stability (BOUT++ elm-pb) v1")
START = time.monotonic(); BUDGET_H = 11.3
TP = Path("/tmp/tp"); RUNS = Path("/tmp/runs"); WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd() / "working"
RES = WORK / "results"; RES.mkdir(parents=True, exist_ok=True); RUNS.mkdir(parents=True, exist_ok=True)
PAYLOAD = "..."
shutil.rmtree(TP, ignore_errors=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as t:
    t.extractall("/tmp", filter="data") if sys.version_info >= (3, 12) else t.extractall("/tmp")
TP = Path("/tmp/tp")
sha = hashlib.sha256((TP / "N9A2_axisymmetric_enlarged_TF.h5").read_bytes()).hexdigest()
assert sha == "8e6b295e4f8dbfe7450975774c580ea9d784483d2afef3ef4de186b121f283f8", "equilibrium file damaged"
print("N9A2 equilibrium SHA-256 OK:", sha)
shutil.copy(TP / "OPEN_IN_PARAVIEW.py", RES / "OPEN_IN_PARAVIEW.py")
print("CPUs:", os.cpu_count())

## 2. Build BOUT++ 5.1.1 + elm-pb

In [ ]:
t0 = time.monotonic()
p = subprocess.Popen(["bash", str(TP / "kaggle_build_bout.sh")], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="", flush=True)
p.wait()
if p.returncode != 0:
    for log in sorted(Path("/tmp/bout/logs").glob("*.log")):
        print("----", log.name); print("".join(log.read_text(errors="replace").splitlines(True)[-25:]))
    raise RuntimeError("BOUT++ build failed (last lines of each log above)")
print(f"BOUT++ + elm-pb built in {(time.monotonic() - t0) / 60:.1f} min")
ENV = dict(os.environ, PATH="/tmp/bout/bin:" + os.environ["PATH"], LD_LIBRARY_PATH="/tmp/bout/lib:" + os.environ.get("LD_LIBRARY_PATH", ""))

## 3. Grids and runs

In [ ]:
CASES = [{"name": "p2_n5_nx260", "n": 5, "nx": 260, "ny": 64, "pscale": 2, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p2_n10_nx260", "n": 10, "nx": 260, "ny": 64, "pscale": 2, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p2_n15_nx260", "n": 15, "nx": 260, "ny": 64, "pscale": 2, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p2_n20_nx260", "n": 20, "nx": 260, "ny": 64, "pscale": 2, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p2_n30_nx260", "n": 30, "nx": 260, "ny": 64, "pscale": 2, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p2_n40_nx260", "n": 40, "nx": 260, "ny": 64, "pscale": 2, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p3_n5_nx260", "n": 5, "nx": 260, "ny": 64, "pscale": 3, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p3_n10_nx260", "n": 10, "nx": 260, "ny": 64, "pscale": 3, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p3_n15_nx260", "n": 15, "nx": 260, "ny": 64, "pscale": 3, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p3_n20_nx260", "n": 20, "nx": 260, "ny": 64, "pscale": 3, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p3_n30_nx260", "n": 30, "nx": 260, "ny": 64, "pscale": 3, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}, {"name": "p3_n40_nx260", "n": 40, "nx": 260, "ny": 64, "pscale": 3, "physics": "spitzer", "opts": ["highbeta:spitzer_resist=true"], "nout": 200, "vtk": true, "est_h": 0.4, "psin0": 0.4, "psin1": 1.0}]
sys.path.insert(0, str(TP / "tools")); sys.path.insert(0, str(TP))
import importlib, bout_results as BR; importlib.reload(BR)
NP = os.cpu_count()
ZIP = WORK / "Tomahawk_Plasma_P2_Results.zip"
def save_zip():
    with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(RES.rglob("*")):
            if f.is_file():
                z.write(f, f.relative_to(WORK))
def grid_for(nx, ny, ps):
    g = TP / "grids" / f"tomahawk_N9A2_nx{nx}_ny{ny}_p{ps}.nc"
    if not g.exists():
        g.parent.mkdir(exist_ok=True)
        r = subprocess.run([sys.executable, str(TP / "tools/make_bout_grid.py"), str(TP / "N9A2_axisymmetric_enlarged_TF.h5"), str(g),
                            "--nx", str(nx), "--ny", str(ny), "--pscale", str(ps)], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("grid failed: " + r.stderr[-2000:])
        d = json.loads(g.with_suffix(".json").read_text())
        print(f"  grid nx={nx} ny={ny} p x{ps}: orthogonality {d['max_abs_cos_angle_x_y_lines_interior']:.1e}, q check {d['ShiftAngle_over_2pi_vs_q_max_rel_diff']:.1e}", flush=True)
    return g
summary, failed = [], []
for c in CASES:
    name = c["name"]; hours_used = (time.monotonic() - START) / 3600
    if hours_used > BUDGET_H - c.get("est_h", 0.5):
        print(f"SKIP {name}: not enough time left ({hours_used:.1f} h used)"); failed.append(name + " (time)"); continue
    print(f"\n=== {name}: n={c['n']} nx={c['nx']} ny={c['ny']} pressure x{c['pscale']} {c['opts']}", flush=True)
    try:
        g = grid_for(c["nx"], c["ny"], c["pscale"])
        d = RUNS / name; shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
        shutil.copy(TP / "BOUT.inp", d / "BOUT.inp"); shutil.copy(g, d / g.name)
        t0 = time.monotonic()
        cmd = ["/tmp/bout/bin/mpiexec", "-n", str(NP), "/tmp/bout/elmpb/build/elm_pb", "-d", ".", f"grid={g.name}", f"zperiod={c['n']}",
               f"nout={c['nout']}", "wall_limit=" + str(max(0.2, BUDGET_H - hours_used - 0.3))] + c["opts"]
        r = subprocess.run(cmd, cwd=d, env=ENV, capture_output=True, text=True)
        wall = time.monotonic() - t0
        if r.returncode != 0 or not list(d.glob("BOUT.dmp.*.nc")):
            raise RuntimeError(f"elm_pb rc={r.returncode}: " + (d / "BOUT.log.0").read_text(errors="replace")[-3000:] if (d / "BOUT.log.0").exists() else r.stderr[-3000:])
        res, P, t, rms = BR.growth_rate(d); res.update(case=c, wall_s=wall)
        (d / "growth_rate.json").write_text(json.dumps(res, indent=1))
        o = RES / name; o.mkdir(exist_ok=True)
        BR.plot_mode(d, d / g.name, o / "mode.png", f"Tomahawk N9A2 p x{c['pscale']}")
        if c.get("vtk", True):
            vdir = BR.export_vtk(d, d / g.name, P, t, every=max(1, len(t) - 1), nphi=48, last_only=True)
            shutil.copytree(vdir, o / "paraview", dirs_exist_ok=True)
        for f in ("growth_rate.json", "BOUT.inp", "BOUT.settings", "BOUT.log.0"):
            shutil.copy(d / f, o / f)
        summary.append({"name": name, **{k: c[k] for k in ("n", "nx", "ny", "pscale")}, "physics": c["physics"],
                        "gamma_per_tau_A": res["gamma_per_alfven_time"], "converged": res["converged"],
                        "peak_psin": c["psin0"] + (c["psin1"] - c["psin0"]) * res["peak_x_index"] / (c["nx"] - 1),
                        "peak_y_index": res["peak_y_index"], "tau_A_s": res["Tbar_s"], "wall_min": wall / 60})
        print(f"  gamma = {res['gamma_per_alfven_time']:.4f} / tau_A (converged: {res['converged']}), wall {wall / 60:.1f} min", flush=True)
        shutil.rmtree(d, ignore_errors=True)
    except Exception as e:
        print("  FAILED:", str(e)[-1500:], flush=True); failed.append(name)
    (RES / "summary.json").write_text(json.dumps({"version": "Tomahawk plasma stability (BOUT++ elm-pb) v1", "cases": summary, "failed": failed}, indent=1))
    save_zip()
print("\nResults zip:", ZIP)
if failed:
    raise RuntimeError(f"{len(failed)} case(s) did not finish: {failed}. Kaggle marks this run failed on purpose.")

## 4. Summary

In [ ]:
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
S = json.loads((RES / "summary.json").read_text())["cases"]
print(f"{'case':38s} {'n':>3s} {'nx':>4s} {'p x':>4s} {'gamma/tau_A':>11s} {'conv':>5s} {'peak psi_n':>10s} {'peak y':>6s} {'min':>6s}")
for s in S:
    print(f"{s['name']:38s} {s['n']:3d} {s['nx']:4d} {s['pscale']:4.1f} {s['gamma_per_tau_A']:11.4f} {str(s['converged']):>5s} {s['peak_psin']:10.3f} {s['peak_y_index']:6d} {s['wall_min']:6.1f}")
fig, ax = plt.subplots(figsize=(8, 5))
for ps in sorted({s["pscale"] for s in S}):
    pts = sorted((s["n"], s["gamma_per_tau_A"]) for s in S if s["pscale"] == ps and s["nx"] == max(x["nx"] for x in S if x["pscale"] == ps))
    if pts:
        ax.plot(*zip(*pts), "o-", label=f"pressure x{ps}")
ax.axhline(0, color="k", lw=0.5); ax.set_xlabel("toroidal mode number n"); ax.set_ylabel("growth rate (1 / tau_A)"); ax.legend()
ax.set_title("Tomahawk N9A2: elm-pb linear growth rates"); plt.tight_layout(); plt.savefig(RES / "growth_vs_n.png", dpi=110)
save_zip(); print("Saved", ZIP)